# Notebook 03 — Vision Agent
Extracts 7 visual features from 3,687 image-backed malicious samples using EasyOCR + OpenCV.

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = str(Path(".").resolve().parent)

DATASET_DIR = os.path.join(BASE, "FINAL_BENCHMARK_DATASET")
IMAGES_DIR  = os.path.join(DATASET_DIR, "IMAGES")
CACHE_DIR   = os.path.join(BASE, "EXPERIMENT_CACHE")
RESULTS_DIR = os.path.join(BASE, "RESULTS")
os.makedirs(CACHE_DIR, exist_ok=True)

# Add AGENTS to path
for p in [os.path.join(BASE, "AGENTS"), os.path.join(Path(".").resolve().parent, "AGENTS")]:
    if p not in sys.path: sys.path.insert(0, p)

print(f"Images dir: {IMAGES_DIR}")
print(f"Images dir exists: {os.path.exists(IMAGES_DIR)}")

In [ ]:
import subprocess
for pkg in ["easyocr", "opencv-python-headless", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("Packages installed.")

from vision_agent import VisionAgent
from ocr_adapter import OCREngineFactory

In [ ]:
# Load all splits, filter to image-backed rows
train_df = pd.read_parquet(os.path.join(DATASET_DIR, "train.parquet"))
val_df   = pd.read_parquet(os.path.join(DATASET_DIR, "validation.parquet"))
test_df  = pd.read_parquet(os.path.join(DATASET_DIR, "test.parquet"))
combined = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Filter to image-backed rows
if "image_path" in combined.columns:
    img_rows = combined[combined["image_path"].notna()].copy()
else:
    print("ERROR: 'image_path' column not found")
    img_rows = combined[combined["label"] == "malicious"].copy()

print(f"Image-backed rows: {len(img_rows):,}")
print(f"Expected: 3,687")

# Resolve full paths
def resolve_image_path(p):
    if p and os.path.isabs(str(p)) and os.path.exists(str(p)):
        return str(p)
    fname = os.path.basename(str(p)) if p else ""
    full = os.path.join(IMAGES_DIR, fname)
    return full if os.path.exists(full) else str(p)

img_rows["full_image_path"] = img_rows["image_path"].apply(resolve_image_path)
exists_mask = img_rows["full_image_path"].apply(os.path.exists)
print(f"Files found on disk: {exists_mask.sum():,} / {len(img_rows):,}")

In [ ]:
# Check for cached results
VISION_CACHE = os.path.join(CACHE_DIR, "vision_features.csv")

if os.path.exists(VISION_CACHE):
    print(f"Loading cached vision features from: {VISION_CACHE}")
    features_df = pd.read_csv(VISION_CACHE)
    print(f"Loaded {len(features_df):,} cached rows.")
else:
    import torch
    GPU = torch.cuda.is_available()
    print(f"Running VisionAgent on {exists_mask.sum():,} images (GPU={GPU})...")
    va = VisionAgent(gpu=GPU)

    valid_rows = img_rows[exists_mask].copy()
    paths  = valid_rows["full_image_path"].tolist()
    sids   = valid_rows["sample_id"].tolist()
    feats  = va.extract_batch(paths, sids, verbose=False)

    features_df = pd.DataFrame([f.to_dict() for f in feats])
    features_df.to_csv(VISION_CACHE, index=False)
    print(f"Vision features saved: {VISION_CACHE} ({len(features_df):,} rows)")

features_df.head(3)

In [ ]:
# Feature distributions by ground truth
feat_cols = ["ocr_confidence", "tiny_text_count", "footer_text_density",
             "watermark_score", "hidden_text_score", "keyword_density", "vision_score"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, feat_cols):
    data = features_df[col].dropna()
    ax.hist(data, bins=30, color="#3498db", alpha=0.8, edgecolor="white")
    ax.axvline(data.mean(), color="#e74c3c", linestyle="--", label=f"mean={data.mean():.3f}")
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Value")
    ax.legend(fontsize=8)
axes.flat[-1].set_visible(False)
plt.suptitle("Vision Feature Distributions (Malicious Samples Only)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "03_vision_feature_distributions.png"),
            bbox_inches="tight", dpi=150)
plt.show()

print("Summary statistics:")
print(features_df[feat_cols].describe().round(4))